In [1]:
import numpy as np

import skimage as ski

In [2]:
img = ski.data.coins()

In [3]:
Hrr, Hrc, Hcc = ski.feature.hessian_matrix(img,
                                           sigma=(3, 4),
                                           order='rc',
                                           use_gaussian_derivatives=True)

/Volumes/zorg/mb312/dev_trees/coordinate-review/main/src/skimage/feature/peak.py:10: ExperimentalAPIWarning: Importing from the `skimage2` namespace is experimental. Its API is under development and considered unstable!
  import skimage2 as ski2


In [4]:
Hxx, Hxy, Hyy = ski.feature.hessian_matrix(img,
                                           sigma=(3, 4),
                                           order='xy',
                                           use_gaussian_derivatives=True)

In [5]:
[np.max(np.abs(a)) for a in (Hxx, Hxy, Hyy)]

[np.float64(0.01456298009396628),
 np.float64(0.007913389098403556),
 np.float64(0.02144621190603369)]

In [6]:
def maxabsd(a1, a2):
    return np.max(np.abs(a1 - a2))

In [7]:
maxabsd(Hrr, Hyy), maxabsd(Hrc, Hxy), maxabsd(Hcc, Hxx)

(np.float64(0.0), np.float64(0.0030066558318056916), np.float64(0.0))

In [8]:
from functools import partial

# Future incantantion.
hessian_matrix = partial(ski.feature.hessian_matrix,
                         use_gaussian_derivatives=True)

def rc_xy_d(func, img, sigma=1):
    Hrr, Hrc, Hcc = func(img, sigma, order='rc')
    Hxx, Hxy, Hyy = func(img, sigma, order='xy')
    return maxabsd(Hrr, Hyy), maxabsd(Hrc, Hxy), maxabsd(Hcc, Hxx)

In [9]:
rc_xy_d(hessian_matrix, img, (3, 4))

(np.float64(0.0), np.float64(0.0030066558318056916), np.float64(0.0))

In [10]:
rc_xy_d(ski.feature.structure_tensor, img)

(np.float64(0.0), np.float64(0.0), np.float64(0.0))

In [11]:
def get_xy_from_rc(func, image, **kwargs):
    """Replicate order='xy' using order='rc' logic.
   
    Function derived from Gemini analysis.
    """
    # Transpose image to reverse axes
    image_t = np.transpose(image)
    
    # Handle sigma if provided as a sequence
    if 'sigma' in kwargs and not np.isscalar(kwargs['sigma']):
        kwargs['sigma'] = kwargs['sigma'][::-1]
        
    # Call the 'rc' version
    res_rc = func(image_t, order='rc', **kwargs)
    
    # Transpose elements back
    return [np.transpose(h) for h in res_rc]

In [12]:
# Replicate 'xy' order with Gemini wrapper.
Hxx_g, Hxy_g, Hyy_g = get_xy_from_rc(hessian_matrix, img, sigma=(3, 4))

In [13]:
maxabsd(Hxx, Hxx_g), maxabsd(Hxy, Hxy_g), maxabsd(Hyy, Hyy_g)

(np.float64(5.204170427930421e-18),
 np.float64(2.6020852139652106e-18),
 np.float64(6.938893903907228e-18))

In [14]:
# Now without use_gaussian_derivatives future argument.
Hxx_0gd, Hxy_0gd, Hyy_0gd = ski.feature.hessian_matrix(
    img,
    sigma=(3, 4),
    order='xy')

/var/folders/hd/rfxyn9gx4bl39bvwzrgn3rtr0000gn/T/ipykernel_22693/2858504691.py:2: FutureWarning: use_gaussian_derivatives currently defaults to False, but will change to True in a future version. Please specify this argument explicitly to maintain the current behavior
  Hxx_0gd, Hxy_0gd, Hyy_0gd = ski.feature.hessian_matrix(


In [15]:
# use_gaussian_derivatives=False makes match between rc / xy worse.
maxabsd(Hrr, Hyy_0gd), maxabsd(Hrc, Hxy_0gd), maxabsd(Hcc, Hxx_0gd)

(np.float64(0.011924403625993568),
 np.float64(0.001534416924632932),
 np.float64(0.0070536815509592635))